In [ ]:
import os
from pathlib import Path
import numpy as np
import scanpy as sc
import muon as mu
from muon import atac as ac

DATA_ROOT = Path(os.environ.get('B_CELL_ATLAS_DATA', 'data'))
sc.set_figure_params(figsize=[6, 6], color_map='viridis_r')

In [ ]:
rna = sc.read_h5ad(DATA_ROOT / 'multimodal_data_RNA.h5ad')
atac = sc.read_h5ad(DATA_ROOT / 'multimodal_data_ATAC.h5ad')
print(rna)
print(atac)

In [ ]:
sc.pp.filter_genes(rna, min_cells=3)
rna.layers['counts'] = rna.X.copy()
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pp.pca(rna)
sc.pp.neighbors(rna, n_neighbors=30, n_pcs=30)
sc.tl.umap(rna)
sc.pl.umap(rna, color='cluster_name', legend_loc='on data')

In [ ]:
rna_glun = rna[rna.obs['in_GluN_trajectory'] == 1].copy()
sc.pp.highly_variable_genes(rna_glun, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pp.pca(rna_glun)
sc.pp.neighbors(rna_glun, n_neighbors=10, n_pcs=20)
sc.tl.umap(rna_glun)
sc.tl.diffmap(rna_glun)
rna_glun.uns['iroot'] = rna_glun.obsm['X_diffmap'][:, 2].argmin()
sc.tl.dpt(rna_glun, n_dcs=5)
sc.pl.umap(rna_glun, color=['cluster_name', 'dpt_pseudotime'])

In [ ]:
sc.pp.calculate_qc_metrics(atac, percent_top=None, log1p=False, inplace=True)

mu.pp.filter_var(atac, 'n_cells_by_counts', lambda x: x >= 30)
ac.pp.binarize(atac)
# TF-IDF normalization for ATAC
ac.pp.tfidf(atac, scale_factor=1e4)
sc.pp.pca(atac)
# store as LSI; first component often heaviy correlates with coverage depth 
atac.obsm['X_lsi'] = atac.obsm['X_pca'].copy()
sc.pp.neighbors(atac, use_rep='X_lsi', n_neighbors=10, n_pcs=30)
sc.tl.umap(atac)
sc.tl.leiden(atac, resolution=0.5)
sc.pl.umap(atac, color='leiden', legend_loc='on data')

In [ ]:
rna.write_h5ad(DATA_ROOT / 'RNA_preprocessed.h5ad')
atac.write_h5ad(DATA_ROOT / 'ATAC_preprocessed.h5ad')